# **Advantages of Cognitive Automation with Generative Models**

Cognitive automation, enhanced by generative models, combines artificial intelligence, machine learning, and advanced data analysis to perform complex tasks with high precision. This approach reduces human errors, standardizes processes, and improves information interpretation, making it especially valuable in sectors such as finance, healthcare, manufacturing, and customer service. In addition to ensuring consistency and reliability in results, it lowers operational costs and rework. With proper training and continuous monitoring, generative models deliver coherent responses, eliminate biases, and strengthen data-driven decision-making.


### Automation of medical processes, specifically for patient triage.

SLM (Small Language Model)不像LLM知道非常廣泛的知識，但是模型體積小、跑很快，很適合具備高度隱私的醫療數據   
RAG (Retrieval-Augmented Generation):讓 AI 去「搜尋」相關的醫學文獻或病歷，看著這些參考資料再來回答問題

In [4]:
# Qdrant 是一個專門儲存向量的資料庫 (Vector Database)，協助我們處理向量資料和相似度搜尋
# gradio 用來展示機器學習或深度學習模型的成果，透過建立網頁形式的使用者介面，提供更直觀的方法去操作功能

!pip install --q qdrant-client gradio transformers
!pip install -U sentence-transformers
!pip install accelerate

In [44]:
# Imports
import torch
import gradio as gr
import pandas as pd
import transformers
import qdrant_client
import sentence_transformers
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams


In [6]:
#For reproduction of the same results as here（跟random_state=42同個道理）

set_seed(1234)

The environment variable below controls tokenization parallelism, i.e., whether multiple threads should be used to process text in parallel. Depending on the environment and workload, enabling or disabling this parallelism can impact performance.

In [8]:
#平行處理Tokenize

%env TOKENIZERS_PARALLELISM=True

env: TOKENIZERS_PARALLELISM=True


## Loading Data to the RAG Module

In [10]:
# Load data into the RAG module

df = pd.read_csv('medquad.csv')
df

,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
...,...,...,...,...
16407,What is (are) Diabetic Neuropathies: The Nerve...,Focal neuropathy appears suddenly and affects ...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...
16408,How to prevent Diabetic Neuropathies: The Nerv...,The best way to prevent neuropathy is to keep ...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...
16409,How to diagnose Diabetic Neuropathies: The Ner...,Doctors diagnose neuropathy on the basis of sy...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...
16410,What are the treatments for Diabetic Neuropath...,The first treatment step is to bring blood glu...,NIDDK,Diabetic Neuropathies: The Nerve Damage of Dia...


In [11]:
# We'll work with just 10000 records to make the app faster.
# Feel free to work with larger data volumes.

ques_data = df['question'].tolist()[:10000]
answer_data = df['answer'].tolist()[:10000]

## Embeddings Model

https://huggingface.co/sentence-transformers/all-mpnet-base-v2

In [13]:
# Defines the embeddings model
# 此模型可以把句子轉成向量 Embeddings

modelo_embedding = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## RAG Module with Vector Database

https://qdrant.tech/

In [15]:
# Use the embeddings model to create vectors from the data
# modelo_embedding（翻譯官）下達encode（編碼指令）就能把問題資料轉成向量

vetores = modelo_embedding.encode(ques_data)

In [16]:
# Create the client for the vector database defined in memory
# 建立一個向量資料庫，並將它建立在記憶體 (RAM)上，這在做 Prototype（原型測試）時是最標準的寫法

banco_vetorial = QdrantClient(":memory:")

In [17]:
# Create the collection in the vector database
# 在資料庫裡面新增收藏（doc_data），並包含兩項規定
# 1. size = len(vetores[0])：每一個向量長度都一樣，都跟第一個向量長度相同
# 2. distance = Distance.COSINE：利用餘弦相似度 (Cosine Similarity)來找最相似的答案

banco_vetorial.create_collection(collection_name = "doc_data",
                                 vectors_config = VectorParams(size = len(vetores[0]),
                                                               distance = Distance.COSINE))

True

In [18]:
# Upload data to the vector database
# upload 資料在新增好的資料庫和收藏之中，利用迴圈把每一份資料貼上id

banco_vetorial.upload_collection(collection_name = "doc_data",
                                 ids = [i for i in range(len(ques_data))],
                                 vectors = vetores)

## Information Retrieval Module

In [20]:
# Define the function that receives a question as input

# 自定義函數，如果病人輸入問題之後，系統必須執行的4個步驟
# 1. 把問題的文字利用SentenceTransformer轉換成向量
# 2. 在剛剛建立的doc_data收藏中query_points（搜尋），這時問題向量會和剛剛的那一萬筆向量做餘弦相似度，並找到分數最高的那幾筆
# 3. 建立sim_ids清單，利用迴圈將剛剛那幾筆得分最高的資料id寫入
# 4. 取sim_ids[0]（得分最高的那一筆資料）

def dsa_recupera_dados(question):

    # Loads the sentence transformer model "all-mpnet-base-v2" from the SentenceTransformer library
    model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

    # Encodes the question into an embedding vector using the model
    ques_vector = model.encode(question)

    # Queries the vector database with the question vector, searching for similar documents
    result = banco_vetorial.query_points(collection_name="doc_data", query=ques_vector)

    # Creates an empty list to store the IDs of the most similar documents
    sim_ids = []

    # Iterates over the query results and adds the document IDs to the list
    for i in result.points:
        sim_ids.append(i.id)

    # Retrieves the context of the most similar document based on the first ID in the list
    context = answer_data[sim_ids[0]]

    # Returns the document context as the answer
    return context

## Integration Module Between SLM and RAG

https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0

In [22]:
# Defines the function 'dsa_llm_rag' that receives a question and a context as input

# SLM：TinyLlama，好處是可以離線在一般電腦上跑，對於高隱私的醫療數據來說很適合
# Prompt Engineering：RAG防止AI產生幻覺 (Hallucination)，只能照著 context 說話
# Tokenization：把上面那串prompt向量化之後，轉成Pytorch(pt)格式的數字矩陣
# Generation：模型開始運作，max_new_tokens=500最多只會回覆500個字，***temperature=1.5：AI的發散、創作力程度
# 模型產出了一堆token_outputs，之後再decode成英文字母，並使用skip_special_tokens=True把奇怪的系統符號省略掉

def dsa_llm_rag(question, context):

    
    # Defines the name of the language model to be used, in this case "TinyLlama-1.1B-Chat-v1.0"
    print("step 1")
    nome_llm = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

    # Loads the tokenizer associated with the model using the specified name
    tokenizer = AutoTokenizer.from_pretrained(nome_llm)

    # Loads the causal language model using the specified name
    # Replace "cuda" with "cpu" if running locally
    model = AutoModelForCausalLM.from_pretrained(nome_llm, device_map="cpu")
    print("done")

    # Defines the app prompt, which includes the question and the context,
    # instructing the LLM to respond based on the context
    chat = [{"role": "user", "content": f"this is question {question} asked by user you are a medical clinic assistant answer the question based on this context {context} in not more than 3-4 points"}]

    # Applies the chat template, tokenizes the prompt and converts it to PyTorch tensors,
    # adding the generation prompt
    # Remove the final [.to("cuda")] part if running locally
    token_inputs = tokenizer.apply_chat_template(chat,
                                                 tokenize=True,
                                                 return_tensors="pt",
                                                 add_generation_prompt=True)

    # Generates the model's response from the input tokens using sampling
    # and setting limits such as the maximum number of new tokens and temperature (creativity level)
    token_outputs = model.generate(input_ids=token_inputs,
                                   do_sample=True,
                                   max_new_tokens=150,
                                   temperature=0.5)

    # Extracts the new tokens generated that are not part of the original input
    new_tokens = token_outputs[0][token_inputs.shape[-1]:]

    # Decodes the new tokens into text, ignoring special tokens
    decoded_output = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # Returns the decoded text as the answer
    return decoded_output

In [23]:
# Defines the function 'dsa_gera_resultado' that receives the user input as a parameter

# 把 R(Retrival)、G(Generation)合併的最終函數：模組化 (Modularization)
# 防呆機制（Edge Case Handling）： if user_input確認是否有打字，沒有的話return "Please enter your question."
# 確認有收到問題後，把user_input丟入剛剛定義的Retrival函數，去向量資料庫尋找最相似的片段，之後將結果存進context變數
# 最後把病人問題和最相似的答案一起丟進剛剛定義的SLM模型中，產出最多500字的回應

def dsa_gera_resultado(user_input):

    # Checks if the user has provided an input
    if user_input:

        # Retrieves the relevant context by calling the 'dsa_recupera_dados' function with the user input
        context = dsa_recupera_dados(user_input)

        # Generates and returns the answer by calling the 'dsa_llm_rag' function with the user input and retrieved context
        return dsa_llm_rag(user_input, context)

    # If the user input is empty, returns a message asking to enter a question
    else:
        return "Please enter your question."

## Web Application Module To Deploy

In [25]:
# Defining the custom interface
webapp = gr.Interface(

    # Function that processes the user input
    fn=dsa_gera_resultado,

    # Larger text box with a custom placeholder
    inputs=gr.Textbox(lines=2, placeholder="Type your question here...", label="Input"),

    # Output text box with a custom label
    outputs=gr.Textbox(label="Answer"),

    # Application title
    title="DSA - Project 3",

    # Custom description
    description="This is an AI application for automating the medical patient triage process.",

    # Pre-defined examples
    examples=[["What is (are) Parasites - Schistosomiasis ?"]],

    # More compact theme
    theme="compact"
)

/opt/anaconda3/lib/python3.12/site-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


In [26]:
# Launch the Gradio interface
webapp.launch(share = False)

/opt/anaconda3/lib/python3.12/site-packages/gradio/utils.py:583: UserWarning: Cannot load compact. Caught Exception: Client error '404 Not Found' for url 'https://huggingface.co/api/spaces/compact' (Request ID: Root=1-6a01dc6b-0ac416dd2c2fac897b6b1c91;f9c0120f-7362-4d04-bdcb-60164a9ee8ca)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404

Sorry, we can't find the page you are looking for.
  warnings.warn(f"Cannot load {theme}. Caught Exception: {str(e)}")


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
